# LangChain 기초 + MLflow 트레이싱
Claude 모델 호출 → 프롬프트 템플릿 → 구조화 출력 → 도구 호출까지.  
모든 호출은 `mlflow.langchain.autolog()` 로 https://mlflow.dove-nest.com 에 트레이스로 남는다.

# **1. 환경 설정**

In [ ]:
from common import setup_mlflow, get_model

setup_mlflow() # MLflow 연결 + autolog

kwargs = {
    'temperature' : 0.6,
}

model = get_model(
        provider   = 'Google',
        model_name = 'gemini-3.1-flash-lite',
        **kwargs
    )

model.model

# **2. 모델 호출**
## **2-1. invoke**
채팅 모델은 메시지 리스트를 받고 `AIMessage` 를 돌려준다. 문자열 하나만 넘기면 HumanMessage 로 감싸진다.

In [2]:
from langchain.messages import SystemMessage, HumanMessage

res = model.invoke([
    SystemMessage("너는 간결하게 답하는 한국어 조교야."),
    HumanMessage("LangChain 과 LangGraph 의 차이를 두 문장으로 설명해줘."),
])
print(res.text)
res.usage_metadata

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


LangChain은 LLM 기반 애플리케이션을 구축하기 위한 다양한 구성 요소와 체인을 제공하는 프레임워크입니다. 반면, LangGraph는 이러한 구성 요소를 순환 구조(Cyclic Graph)로 연결하여 복잡한 에이전트 워크플로우를 제어하고 관리하는 데 특화된 도구입니다.


{'input_tokens': 30,
 'output_tokens': 74,
 'total_tokens': 104,
 'input_token_details': {'cache_read': 0}}

## **2-2. stream**

In [3]:
for chunk in model.stream("RAG 가 뭔지 세 줄로 설명해줘."):
    print(chunk.text, end="", flush=True)

RAG(검색 증강 생성)에 대해 세 줄로 설명해 드립니다.

1. **외부 데이터 참조:** AI가 학습하지 않은 최신 정보나 내부 문서를 실시간으로 검색하여 답변의 근거로 활용하는 기술입니다.
2. **환각 현상 방지:** 모델이 모르는 내용을 지어내는 '환각(Hallucination)'을 줄이고, 신뢰도 높은 정확한 답변을 제공합니다.
3. **효율적 최신화:** 모델 자체를 매번 재학습시킬 필요 없이, 참조하는 데이터베이스만 업데이트하여 항상 최신 상태를 유지할 수 있습니다.

# **3. 프롬프트 템플릿과 체인 (LCEL)**
`|` 로 Runnable 을 이어 붙인다. MLflow 트레이스에서 각 단계가 span 으로 보인다.

In [4]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_messages([
    ("system", "너는 {domain} 전문가야. 초보자에게 설명하듯 답해."),
    ("human", "{question}"),
])
chain = prompt | model | StrOutputParser()

print(chain.invoke({"domain": "데이터베이스", "question": "인덱스는 왜 쓰나요?"}))

데이터베이스에서 **'인덱스(Index)'**를 아주 쉽게 설명해 드릴게요.

가장 좋은 비유는 **'두꺼운 전공 서적 맨 뒤에 있는 찾아보기(색인)'**입니다.

---

### 1. 인덱스가 없다면? (책 전체를 다 읽기)
만약 500페이지짜리 책에서 '데이터베이스'라는 단어가 어디에 나오는지 찾고 싶다고 해볼게요. 인덱스가 없다면, 1페이지부터 500페이지까지 한 글자씩 다 뒤져야 합니다. 
*   **컴퓨터 관점:** 데이터베이스의 모든 행을 처음부터 끝까지 다 뒤지는 작업을 **'Full Table Scan'**이라고 합니다. 데이터가 100만 건이라면 시간이 엄청나게 걸리겠죠?

### 2. 인덱스가 있다면? (찾아보기 활용)
책 맨 뒤의 '찾아보기'에서 '데이터베이스'를 찾습니다. 거기엔 "123페이지, 450페이지"라고 적혀 있죠. 그럼 우리는 바로 그 페이지만 펼치면 됩니다.
*   **컴퓨터 관점:** 인덱스는 데이터의 위치 정보를 미리 정렬해서 저장해둔 별도의 표입니다. 그래서 원하는 데이터를 찾을 때 훨씬 빠르게 접근할 수 있습니다.

---

### 3. 왜 인덱스를 쓰나요? (핵심 이유)
한마디로 **'검색 속도 향상'** 때문입니다.

*   **조회(SELECT)가 빨라집니다:** 수백만 건의 데이터 중에서 특정 조건을 만족하는 데이터를 0.1초 만에 찾아낼 수 있게 해줍니다.
*   **정렬(ORDER BY)이나 그룹화(GROUP BY)가 빨라집니다:** 인덱스 자체가 이미 정렬된 상태로 저장되기 때문에, 데이터를 다시 정렬할 필요가 없어 성능이 좋아집니다.

---

### 4. 주의할 점 (공짜는 아니다!)
"그럼 모든 컬럼에 인덱스를 다 걸면 되겠네?"라고 생각할 수 있지만, 그러면 안 됩니다. 왜냐하면 **'부작용'**이 있기 때문입니다.

1.  **저장 공간 차지:** 인덱스도 결국 데이터베이스 안에 저장되는 별도의 파일입니다. 너무 많이 만들면 용량을 엄청나게 잡아먹습니다.
2.  **데이터 수정(INSERT, UPD

# **4. 구조화 출력**
Pydantic 스키마를 주면 파싱된 객체로 받는다.

In [5]:
from pydantic import BaseModel, Field

class Paper(BaseModel):
    title: str = Field(description="논문 제목")
    year: int = Field(description="발표 연도")
    key_idea: str = Field(description="핵심 아이디어 한 문장")

structured = model.with_structured_output(Paper)
paper = structured.invoke("Attention Is All You Need 논문 정보를 알려줘.")
paper

Paper(title='Attention Is All You Need', year=2017, key_idea='RNN이나 CNN 없이 오직 어텐션 메커니즘만을 사용하여 병렬 처리가 가능하고 성능이 뛰어난 트랜스포머 아키텍처를 제안함')

# **5. 도구 호출 (Tool calling)**
`@tool` 로 함수를 도구로 만들고 `bind_tools` 로 모델에 연결한다.  
모델은 도구를 **직접 실행하지 않고** 호출 요청(`tool_calls`)만 돌려준다. 실행 루프는 02 노트북에서 LangGraph 로 만든다.

In [6]:
from langchain.tools import tool

@tool
def multiply(a: float, b: float) -> float:
    """두 수를 곱한다."""
    return a * b

@tool
def get_weather(city: str) -> str:
    """도시의 현재 날씨를 알려준다."""
    return f"{city}: 맑음, 21도"  # 실습용 가짜 데이터

model_with_tools = model.bind_tools([multiply, get_weather])
ai = model_with_tools.invoke("서울 날씨 알려주고, 37.5 곱하기 4 도 계산해줘.")
ai.tool_calls

[{'name': 'get_weather',
  'args': {'city': '서울'},
  'id': 'call_625830',
  'type': 'tool_call'},
 {'name': 'multiply',
  'args': {'b': 4, 'a': 37.5},
  'id': 'call_625831',
  'type': 'tool_call'}]

In [7]:
# 도구를 직접 실행하고 결과를 돌려주면 최종 답변을 받는다
from langchain.messages import HumanMessage

tools = {"multiply": multiply, "get_weather": get_weather}
messages = [HumanMessage("서울 날씨 알려주고, 37.5 곱하기 4 도 계산해줘."), ai]
for call in ai.tool_calls:
    messages.append(tools[call["name"]].invoke(call))  # ToolMessage 반환

print(model_with_tools.invoke(messages).text)

현재 서울의 날씨는 맑고 기온은 21도입니다. 또한, 37.5 곱하기 4의 결과는 150입니다.


# **6. MLflow 에서 트레이스 확인**
UI: https://mlflow.dove-nest.com → Experiments → `langchain-study` → **Traces** 탭  
코드로도 조회할 수 있다.

In [8]:
import mlflow

traces = mlflow.search_traces(max_results=10)
traces[["trace_id", "request_time", "state", "execution_duration"]]

,trace_id,request_time,state,execution_duration
0,tr-cfc9cb0fd444795513a7a79fa158a743,1790584512026,OK,1157
1,tr-27d77b9f3f89ec81c9c402af88b0a17e,1790584512024,OK,1
2,tr-c9da660b3e28c50b2f7c64eaf6a02b86,1790584512022,OK,1
3,tr-445e298dd36ae38eb3d4683a237efb6e,1790584511260,OK,752
4,tr-91f05bea56df0339a6638c2257240560,1790584510046,OK,1141
5,tr-fc0d21da978c50b32ef968d572937971,1790584506668,OK,3365
6,tr-d33242c7df8fd322b0bab5f30b9edfea,1790584504932,OK,1674
7,tr-cc2937adc817efd6cee3593cfd4a8946,1790584503491,OK,1433
8,tr-0864741be8e97eda6af4fd2694c48650,1790584321855,OK,1219
9,tr-f2f687e2ad20ac182abfb89558446b37,1790584321853,OK,1


# **99. 참고자료**
- LangChain 문서: https://docs.langchain.com/oss/python/langchain/overview
- ChatAnthropic: https://docs.langchain.com/oss/python/integrations/chat/anthropic
- MLflow LangChain 트레이싱: https://mlflow.org/docs/latest/genai/tracing/integrations/listing/langchain